# RetailOps AI
## Notebook 05 — Context Engineering and Harness Controls

This notebook adds the remaining context and harness controls required by the capstone:

- AGENTS.md system specification
- Context compaction rule
- Tool-result size limit
- Step budget
- Cost budget
- Audit logging

Policy retrieval was implemented in Notebook 02.
Human approval for write actions was implemented in Notebook 04.

Cell 2 — Setup

In [1]:
!pip -q install openai

In [2]:
from pathlib import Path
import json
import os
import sys
import subprocess

SDAIA_DIR = Path("/content/agentic-ai")

if not SDAIA_DIR.exists():
    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/SDAIAAcademy/agentic-ai.git",
            str(SDAIA_DIR)
        ],
        check=True
    )

sys.path.insert(0, str(SDAIA_DIR / "labs"))

import agentkit as K

WORK_DIR = Path("/content/RetailOps_AI")
WORK_DIR.mkdir(exist_ok=True)

print("✅ Harness environment ready.")

✅ Harness environment ready.


Cell 3 — Create AGENTS.md

In [3]:
AGENTS_PATH = WORK_DIR / "AGENTS.md"

AGENTS_TEXT = """
# RetailOps AI Agent Specification

## Role
RetailOps AI is a policy-aware retail customer-service agent.

## Primary Goals
1. Identify the customer before accessing private order information.
2. Use tools instead of guessing customer, order, product, or policy data.
3. Verify that an order belongs to the identified customer.
4. Retrieve relevant retail policy before making policy-dependent decisions.
5. Keep responses concise, factual, and grounded in tool results.

## Read Actions
Read-only tools may execute automatically.

Examples:
- find_user_by_email
- get_user_details
- get_order_details
- get_product_details
- retrieve_policy

## Write Actions
Any state-changing action requires explicit human approval before execution.

Examples:
- cancel_pending_order

## Safety Rules
- Never invent order status, product availability, customer details, or policy rules.
- Never expose an order belonging to another customer.
- Never perform a write action without approval.
- If required information is missing, ask for it.
- Tool errors must be reported honestly.

## Context Rules
- Keep only the most recent relevant conversation history.
- Limit large tool outputs before adding them to model context.
- Prefer retrieved policy sections instead of loading the entire policy document.

## Harness Budgets
- Maximum agent steps: 8
- Maximum run cost: $0.05
- Maximum tool-result size: 3000 characters
- Maximum conversation history: 24 messages
"""

AGENTS_PATH.write_text(
    AGENTS_TEXT.strip(),
    encoding="utf-8"
)

print("✅ AGENTS.md created.")
print(AGENTS_PATH)

✅ AGENTS.md created.
/content/RetailOps_AI/AGENTS.md


Cell 4 — Load Agent Specification

In [4]:
SYSTEM_SPEC = AGENTS_PATH.read_text(encoding="utf-8")

print(SYSTEM_SPEC[:1000])

# RetailOps AI Agent Specification

## Role
RetailOps AI is a policy-aware retail customer-service agent.

## Primary Goals
1. Identify the customer before accessing private order information.
2. Use tools instead of guessing customer, order, product, or policy data.
3. Verify that an order belongs to the identified customer.
4. Retrieve relevant retail policy before making policy-dependent decisions.
5. Keep responses concise, factual, and grounded in tool results.

## Read Actions
Read-only tools may execute automatically.

Examples:
- find_user_by_email
- get_user_details
- get_order_details
- get_product_details
- retrieve_policy

## Write Actions
Any state-changing action requires explicit human approval before execution.

Examples:
- cancel_pending_order

## Safety Rules
- Never invent order status, product availability, customer details, or policy rules.
- Never expose an order belonging to another customer.
- Never perform a write action without approval.
- If required informat

Cell 5 — Harness Budgets

In [5]:
MAX_STEPS = 8
MAX_COST = 0.05
MAX_RESULT_CHARS = 3000
MAX_HISTORY_MESSAGES = 24

print("Harness budgets")
print("----------------")
print("Max steps:", MAX_STEPS)
print("Max cost: $", MAX_COST)
print("Max tool result chars:", MAX_RESULT_CHARS)
print("Max history messages:", MAX_HISTORY_MESSAGES)

Harness budgets
----------------
Max steps: 8
Max cost: $ 0.05
Max tool result chars: 3000
Max history messages: 24


Cell 6 — Context Compaction

In [6]:
def compact_history(history):

    if len(history) <= MAX_HISTORY_MESSAGES:
        return history

    compacted = history[-MAX_HISTORY_MESSAGES:]

    # Avoid beginning with an orphan tool result.
    while compacted and compacted[0].get("role") not in {
        "user",
        "assistant"
    }:
        compacted = compacted[1:]

    return compacted

Cell 7 — Test Compaction

In [7]:
demo_history = []

for i in range(40):
    demo_history.append({
        "role": "user" if i % 2 == 0 else "assistant",
        "content": f"Message {i + 1}"
    })

compacted = compact_history(demo_history)

print("Before:", len(demo_history))
print("After :", len(compacted))

assert len(compacted) <= MAX_HISTORY_MESSAGES

print("✅ Context compaction enforced.")

Before: 40
After : 24
✅ Context compaction enforced.


Cell 8 — Audit Log + Tool Result Cap

In [8]:
AUDIT_PATH = WORK_DIR / "audit.jsonl"


def after_tool(name, arguments, result):

    result = str(result)

    record = {
        "tool": name,
        "arguments": arguments,
        "result_chars": len(result)
    }

    with AUDIT_PATH.open(
        "a",
        encoding="utf-8"
    ) as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

    if len(result) > MAX_RESULT_CHARS:

        trimmed = result[:MAX_RESULT_CHARS]

        trimmed += (
            f"\n[… {len(result) - MAX_RESULT_CHARS} "
            "characters removed by context budget]"
        )

        return trimmed

    return result

Cell 9 — Verify Tool Result Cap

In [9]:
large_result = "A" * 5000

capped_result = after_tool(
    "demo_tool",
    {"query": "test"},
    large_result
)

print("Original chars:", len(large_result))
print("Capped chars:", len(capped_result))

assert len(capped_result) < len(large_result)
assert "characters removed" in capped_result

print("✅ Tool-result budget enforced.")

Original chars: 5000
Capped chars: 3046
✅ Tool-result budget enforced.


Cell 10 — Verify Audit Log

In [10]:
audit_records = [
    json.loads(line)
    for line in AUDIT_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
]

audit_records[-1]

{'tool': 'demo_tool', 'arguments': {'query': 'test'}, 'result_chars': 5000}

In [11]:
{
    'tool': 'demo_tool',
    'arguments': {'query': 'test'},
    'result_chars': 5000
}

{'tool': 'demo_tool', 'arguments': {'query': 'test'}, 'result_chars': 5000}

Cell 11 — Permission Policy

In [12]:
WRITE_TOOLS = {
    "cancel_pending_order"
}


def permission(tool_name, arguments):

    if tool_name in WRITE_TOOLS:
        return "ask"

    return "allow"

In [13]:
assert permission(
    "get_order_details",
    {}
) == "allow"

assert permission(
    "cancel_pending_order",
    {}
) == "ask"

print("✅ Permission policy enforced.")

✅ Permission policy enforced.


Cell 12 — Step + Cost Budget Harness

In [14]:
print("K.run_agent supports:")

print("• max_steps =", MAX_STEPS)
print("• max_cost  =", MAX_COST)

print("\n✅ Step and cost budgets are configured for every agent run.")

K.run_agent supports:
• max_steps = 8
• max_cost  = 0.05

✅ Step and cost budgets are configured for every agent run.


In [15]:
def run_retail_agent(
    task,
    tools,
    history=None,
    approve=None
):

    return K.run_agent(
        task,
        tools,
        system=SYSTEM_SPEC,
        history=compact_history(history or []),
        max_steps=MAX_STEPS,
        max_cost=MAX_COST,
        permission=permission,
        approve=approve,
        after_tool=after_tool,
        verbose=False
    )

In [16]:
max_steps=MAX_STEPS
max_cost=MAX_COST

Cell 13 — Harness Summary

In [17]:
harness_summary = {
    "context_file": AGENTS_PATH.exists(),
    "retrieval": "Implemented in Notebook 02",
    "compaction": True,
    "tool_result_budget": MAX_RESULT_CHARS,
    "step_budget": MAX_STEPS,
    "cost_budget_usd": MAX_COST,
    "human_approval": "Implemented in Notebook 04",
    "audit_logging": AUDIT_PATH.exists()
}

harness_summary

{'context_file': True,
 'retrieval': 'Implemented in Notebook 02',
 'compaction': True,
 'tool_result_budget': 3000,
 'step_budget': 8,
 'cost_budget_usd': 0.05,
 'human_approval': 'Implemented in Notebook 04',
 'audit_logging': True}

In [18]:
{
    'context_file': True,
    'retrieval': 'Implemented in Notebook 02',
    'compaction': True,
    'tool_result_budget': 3000,
    'step_budget': 8,
    'cost_budget_usd': 0.05,
    'human_approval': 'Implemented in Notebook 04',
    'audit_logging': True
}

{'context_file': True,
 'retrieval': 'Implemented in Notebook 02',
 'compaction': True,
 'tool_result_budget': 3000,
 'step_budget': 8,
 'cost_budget_usd': 0.05,
 'human_approval': 'Implemented in Notebook 04',
 'audit_logging': True}

Cell 14 — Final Requirement Check

In [19]:
assert AGENTS_PATH.exists()
assert len(compacted) <= MAX_HISTORY_MESSAGES
assert MAX_STEPS == 8
assert MAX_COST == 0.05
assert MAX_RESULT_CHARS == 3000
assert AUDIT_PATH.exists()

print("✅ AGENTS.md")
print("✅ Context compaction")
print("✅ Tool-result budget")
print("✅ Step budget")
print("✅ Cost budget")
print("✅ Audit logging")
print("✅ Permission policy")

print("\nNotebook 05: PASS")

✅ AGENTS.md
✅ Context compaction
✅ Tool-result budget
✅ Step budget
✅ Cost budget
✅ Audit logging
✅ Permission policy

Notebook 05: PASS


## Conclusion

RetailOps AI now has explicit context and harness controls.

Implemented controls:

- AGENTS.md system specification
- policy retrieval from Notebook 02
- conversation compaction
- capped tool outputs
- allow/ask permission policy
- human approval from Notebook 04
- maximum step budget
- maximum cost budget
- audit logging

These controls complete the capstone's core context-engineering and harness requirements.

### Next

Notebook 06 will implement the evaluation suite:

- held-out evaluation tasks
- automatic graders
- Configuration A vs B
- quantitative metrics
- error analysis